# 02 Check Data DIBaS

This notebook audits the original DIBaS slides and writes the clean slide manifest used by the later notebooks.


In [1]:
from collections import Counter
from pathlib import Path
import csv
import json

import pandas as pd

from IPython.display import display
from PIL import Image


# This helper keeps the notebook easy to run from the repo root or from notebooks/.
def find_repo_root(start_path: Path) -> Path:
    if (start_path / "raw_data").exists():
        return start_path
    if start_path.name == "notebooks" and (start_path.parent / "raw_data").exists():
        return start_path.parent
    raise FileNotFoundError("Could not find the FYP2 repo root.")

# This configuration cell keeps the main paths and audit settings in one visible place.
REPO_ROOT = find_repo_root(Path.cwd())
RAW_DATA_DIR = REPO_ROOT / "raw_data"
MANIFESTS_DIR = REPO_ROOT / "manifests"
RESULTS_DIR = REPO_ROOT / "results"
NOTEBOOK_TAG = "02_check_data_dibas"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / NOTEBOOK_TAG
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


SPECIES_TO_BINARY = {
    "Staphylococcus.aureus": 0,
    "Enterococcus.faecalis": 0,
    "Enterococcus.faecium": 0,
    "Escherichia.coli": 1,
    "Listeria.monocytogenes": 1,
    "Clostridium.perfringens": 1,
    "Pseudomonas.aeruginosa": 1,
}

EXPECTED_SLIDE_COUNTS = {
    "Clostridium.perfringens": 23,
    "Enterococcus.faecalis": 20,
    "Enterococcus.faecium": 20,
    "Escherichia.coli": 20,
    "Listeria.monocytogenes": 22,
    "Pseudomonas.aeruginosa": 20,
    "Staphylococcus.aureus": 20,
}
EXPECTED_IMAGE_SIZE = (2048, 1532)

print(f"Repo root: {REPO_ROOT}")
print(f"Notebook tag: {NOTEBOOK_TAG}")

Repo root: C:\Users\FYP2610\Downloads\FYP2
Notebook tag: 02_check_data_dibas


In [2]:
# This helper writes a CSV file with a stable header.
def write_csv_rows(csv_path: Path, rows, fieldnames):
    if not rows:
        raise ValueError(f"No rows were provided for {csv_path.name}.")
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

# This helper writes one small JSON file with clean formatting.
def write_json(json_path: Path, data):
    json_path.parent.mkdir(parents=True, exist_ok=True)
    json_path.write_text(json.dumps(data, indent=2), encoding="utf-8")

# This helper stores paths in repo-relative form so the outputs stay portable.
def as_repo_relative(repo_root: Path, path_value: Path) -> str:
    return path_value.relative_to(repo_root).as_posix()

In [3]:
# We scan every TIFF file and store the exact slide path, species, image size, and binary label.
slide_rows = []
for species_name in sorted(SPECIES_TO_BINARY):
    species_dir = RAW_DATA_DIR / species_name
    slide_paths = sorted(species_dir.glob("*.tif"))
    for slide_path in slide_paths:
        with Image.open(slide_path) as image_handle:
            width, height = image_handle.size

        slide_rows.append({
            "slide_id": slide_path.stem,
            "species_name": species_name,
            "binary_label": SPECIES_TO_BINARY[species_name],
            "binary_group": "cocci" if SPECIES_TO_BINARY[species_name] == 0 else "bacilli",
            "raw_path": as_repo_relative(REPO_ROOT, slide_path),
            "width": width,
            "height": height,
        })

if len(slide_rows) != 145:
    raise AssertionError(f"Expected 145 DIBaS slides, but found {len(slide_rows)}.")

species_counts = Counter(row["species_name"] for row in slide_rows)
size_counts = Counter((row["width"], row["height"]) for row in slide_rows)
binary_counts = Counter(row["binary_group"] for row in slide_rows)

if dict(species_counts) != EXPECTED_SLIDE_COUNTS:
    raise AssertionError(f"Species counts do not match the expected DIBaS subset: {species_counts}")
if set(size_counts) != {EXPECTED_IMAGE_SIZE}:
    raise AssertionError(f"Unexpected DIBaS image sizes were found: {size_counts}")

slide_manifest_path = MANIFESTS_DIR / "slide_manifest.csv"
write_csv_rows(slide_manifest_path, slide_rows, list(slide_rows[0].keys()))

species_rows = [{"species_name": key, "slide_count": species_counts[key]} for key in sorted(species_counts)]
binary_rows = [{"binary_group": key, "slide_count": binary_counts[key]} for key in sorted(binary_counts)]
size_rows = [{"width": key[0], "height": key[1], "slide_count": value} for key, value in sorted(size_counts.items())]

write_csv_rows(NOTEBOOK_RESULTS_DIR / "species_counts.csv", species_rows, list(species_rows[0].keys()))
write_csv_rows(NOTEBOOK_RESULTS_DIR / "binary_counts.csv", binary_rows, list(binary_rows[0].keys()))
write_csv_rows(NOTEBOOK_RESULTS_DIR / "image_size_counts.csv", size_rows, list(size_rows[0].keys()))

summary = {
    "notebook_tag": NOTEBOOK_TAG,
    "slide_count": len(slide_rows),
    "species_count": len(species_counts),
    "cocci_slide_count": binary_counts["cocci"],
    "bacilli_slide_count": binary_counts["bacilli"],
    "expected_image_size": EXPECTED_IMAGE_SIZE,
    "slide_manifest_path": as_repo_relative(REPO_ROOT, slide_manifest_path),
}
write_json(NOTEBOOK_RESULTS_DIR / "summary.json", summary)

display(pd.DataFrame(slide_rows[:5]))
display(pd.DataFrame(species_rows))
display(pd.DataFrame(binary_rows))
display(pd.DataFrame(size_rows))
print(f"Saved slide manifest to: {slide_manifest_path}")

,slide_id,species_name,binary_label,binary_group,raw_path,width,height
0,Clostridium.perfringens_0001,Clostridium.perfringens,1,bacilli,raw_data/Clostridium.perfringens/Clostridium.p...,2048,1532
1,Clostridium.perfringens_0002,Clostridium.perfringens,1,bacilli,raw_data/Clostridium.perfringens/Clostridium.p...,2048,1532
2,Clostridium.perfringens_0003,Clostridium.perfringens,1,bacilli,raw_data/Clostridium.perfringens/Clostridium.p...,2048,1532
3,Clostridium.perfringens_0004,Clostridium.perfringens,1,bacilli,raw_data/Clostridium.perfringens/Clostridium.p...,2048,1532
4,Clostridium.perfringens_0005,Clostridium.perfringens,1,bacilli,raw_data/Clostridium.perfringens/Clostridium.p...,2048,1532


,species_name,slide_count
0,Clostridium.perfringens,23
1,Enterococcus.faecalis,20
2,Enterococcus.faecium,20
3,Escherichia.coli,20
4,Listeria.monocytogenes,22
5,Pseudomonas.aeruginosa,20
6,Staphylococcus.aureus,20


,binary_group,slide_count
0,bacilli,85
1,cocci,60


,width,height,slide_count
0,2048,1532,145


Saved slide manifest to: C:\Users\FYP2610\Downloads\FYP2\manifests\slide_manifest.csv
